[Lab README](README.md)

# Lab 5.1: Deploy the Lab 4 agent to AgentCore Runtime

The agent from Lab 4 runs on your machine. This notebook puts the same agent on
Amazon Bedrock AgentCore Runtime, invokes it four times over the network, and
leaves you an ARN. The retrieval tool does not change: `booking_agent.py`
imports `search_hotel_knowledge` from the same `workshop.hybrid_retrieval` that
Labs 2, 3 and 4 imported. That is the claim this notebook is here to test, and
the four smoke tests at the end are how it gets tested. Each one asserts, so a
deployed agent that answers without reaching the graph, or a reservation write
that lands twice, fails the cell instead of printing something plausible.

## The two AgentCore services this lab uses

Both are new here, so it is worth being precise about what each one is before
the notebook starts calling them.

**AgentCore Runtime** is managed hosting for a single agent. You give it a
container image and an execution role. It runs the container, gives it an ARN to
be invoked by, isolates every invocation in its own session, and forwards the
container's logs and traces to CloudWatch. It is not a model and it is not an
agent framework. The Strands agent inside the container is the same object Lab 3
built; what Runtime adds is the address, the isolation, and the operational
surface around it.

**AgentCore Gateway** turns an existing AWS resource into an MCP tool. Here it
fronts one Lambda, the reservation command, and publishes it over MCP so the
agent discovers the tool at startup rather than having it compiled in. The agent
never learns it is calling a Lambda. It sees a tool with a schema.

The two are independent of each other. Runtime is where the agent lives, Gateway
is how the agent reaches a capability it does not own.

One thing does change between Lab 4 and here. Locally, the reservation command
was a Python function the agent called in process. Deployed, it is a Lambda
behind the Gateway, discovered as an MCP tool. The agent code is identical
either way, because both forms satisfy the same frozen five-field contract.

**This notebook creates billable AWS resources.** An ECR repository, a CodeBuild
project, and an AgentCore Runtime. `5.2_teardown.ipynb` deletes them, and
nothing else in the workshop does. Run it before you stop for the day.

**Prerequisites**

1. Lab 1 has built the graph, including the Cairo hero `Hotel` and the
   `max_guests` rule. The deployed agent reads that graph, not a copy of it.
2. `setup/provision_agentcore.py provision` has run. It creates the Gateway,
   the reservation Lambda, the Neo4j command secret, and the Runtime execution
   role, and writes three values into the repository-root `.env`. Step 1 checks
   for them and tells you what to run if they are missing.
3. AWS credentials with Bedrock model access in `AWS_REGION`, and Neo4j
   credentials in the repository-root `.env`.
4. **One AWS account per participant, or one `DEMO06_PREFIX` per participant.**
   Every name this lab creates is built from that prefix, which defaults to
   `demo06`. Two people sharing an account on the default prefix launch the same
   Runtime under the same name, and the second launch overwrites the first
   without saying so. Export a unique prefix before you provision and before you
   run this notebook, and both halves stay yours:

   ```bash
   export DEMO06_PREFIX=demo06-yourname
   ```

   The Gateway created for you carries no authorizer, to keep the lab short. Its
   URL is a credential: anyone holding it can write `ReservationRequest` nodes
   into your graph. Keep it out of screen shares, and run both teardowns when
   you are done.

Without credentials every live cell below skips and the notebook still runs
clean. That is how repository validation passes offline.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("Environment ready")

---
## Step 1: Confirm the managed boundary already exists

`provision_agentcore.py` builds the half of this deployment that this notebook
does not: the Gateway that fronts the reservation command, the Lambda behind it,
the secret that Lambda reads Neo4j from, and the IAM role the Runtime assumes.
Splitting it that way keeps this notebook to one job, deploying the agent, and
keeps the resource creation in a script you can read, re-run, and tear down from
a terminal.

Three values land in the repository-root `.env`:

| Value | What it is |
| --- | --- |
| `AGENTCORE_GATEWAY_URL` | The MCP endpoint the deployed agent discovers `create_reservation_request` from |
| `AGENTCORE_RUNTIME_ROLE_ARN` | The execution role the Runtime assumes |
| `NEO4J_COMMAND_SECRET_ID` | The Secrets Manager entry the reservation Lambda reads its Neo4j write credential from |

The Runtime itself reads Neo4j from environment variables rather than from a
secret, so its role grants no secret access at all. That is a deliberate
narrowing: the retrieval path is read-only and its credential is passed in as
container configuration, while the one path that writes to the graph reads a
credential the Runtime cannot see.

### The shape of the command secret

`NEO4J_COMMAND_SECRET_ID` points at a Secrets Manager secret whose value is a
JSON document with exactly these four keys:

```json
{
  "uri": "neo4j+s://<id>.databases.neo4j.io",
  "username": "neo4j",
  "password": "...",
  "database": "neo4j"
}
```

Worth knowing before you need it. If the reservation command fails with a Neo4j
authentication error, the Lambda read that secret and got something it could not
use, and the four keys are the first thing to check. A secret stored as a plain
string, or with `user` instead of `username`, produces an auth failure that
looks like a bad password. Read the secret's value in the Secrets Manager
console and compare it against the shape above. Re-running
`uv run setup/provision_agentcore.py provision` rewrites the secret from the
`NEO4J_*` values in the repository-root `.env`, which is usually the fix.

In [ ]:
import os
import re
import shutil
import subprocess
from pathlib import Path

import boto3
from dotenv import load_dotenv

from workshop.bedrock_providers import default_model_id

# The repository-root `.env` is where provision_agentcore.py writes, and where
# the NEO4J_* values every other lab uses already live. A folder-local `.env`
# wins if a participant made one.
#
# The marker is setup/run_notebooks.py rather than README.md, because every lab
# folder has a README.md and the walk up would stop at the first one.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "setup" / "run_notebooks.py").is_file():
    if REPO_ROOT == REPO_ROOT.parent:
        raise FileNotFoundError(
            "Repository root not found. Run this notebook from 05-agentcore-deploy/."
        )
    REPO_ROOT = REPO_ROOT.parent
load_dotenv()
load_dotenv(REPO_ROOT / ".env")

REGION = os.environ.get("AWS_REGION", "us-east-1")
GATEWAY_URL = os.getenv("AGENTCORE_GATEWAY_URL", "").strip()
RUNTIME_ROLE_ARN = os.getenv("AGENTCORE_RUNTIME_ROLE_ARN", "").strip()

# One definition of the model id, in the shared package, applying the same
# MODEL_ID override every other lab honors. Restating the literal here would put
# a second copy in the tree, and the copy that drifts is the one that decides
# what the deployed agent runs on.
MODEL_ID = default_model_id()

# The deployed Runtime reads its read-only Neo4j connection from these, so they
# have to be forwarded as container environment variables at launch.
NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_VALUES = {name: os.getenv(name, "").strip() for name in NEO4J_ENV}

# Every name this deploy uses comes from one prefix.
#
# `demo06` is the real provisioned prefix and the default. Set DEMO06_PREFIX to
# something unique before running `setup/provision_agentcore.py provision` and
# before running this notebook, and you get your own Gateway, Lambda, secret,
# roles, Runtime, ECR repository and CodeBuild project. That is what makes it
# safe for several participants to share one AWS account: with the default
# prefix they all launch the same Runtime name, and the starter toolkit's
# auto_update_on_conflict overwrites rather than complains.
#
# Both values below have to be derived rather than written out. The Runtime name
# decides the ECR repository and CodeBuild project names, and workshop_cleanup.py
# derives the same three from the same variable, so teardown can only find what
# this notebook created if both files agree. The Gateway target name is half of
# the MCP tool name, and the deployed agent refuses to start against a Gateway
# that publishes anything else, which is why it is passed into the container at
# launch alongside GATEWAY_URL.
#
# AgentCore Runtime names take letters, digits and underscores only, so a prefix
# with a hyphen in it is folded into CamelCase.
DEMO06_PREFIX = os.environ.get("DEMO06_PREFIX", "").strip() or "demo06"
RUNTIME_SUFFIX = (
    ""
    if DEMO06_PREFIX == "demo06"
    else "".join(
        part.capitalize() for part in re.split(r"[^A-Za-z0-9]+", DEMO06_PREFIX) if part
    )
)
RUNTIME_NAME = f"HotelBookingAgent{RUNTIME_SUFFIX}"
if len(RUNTIME_NAME) > 48:
    raise ValueError(
        f"DEMO06_PREFIX makes a Runtime name of {len(RUNTIME_NAME)} characters; "
        "AgentCore allows 48. Choose a shorter prefix."
    )
GATEWAY_TARGET_NAME = f"{DEMO06_PREFIX}-reservation-request"

# The teardown tag gate. workshop_cleanup.py deletes a resource only if it
# carries this exact key and value; a near-miss is the same as no tag at all.
# The three shapes below are not interchangeable, each service demands its own.
WORKSHOP_TAG_KEY = "WorkshopResource"
WORKSHOP_TAG_VALUE = "stop-ai-agent-hallucinations"
WORKSHOP_TAGS_MAP = {WORKSHOP_TAG_KEY: WORKSHOP_TAG_VALUE}                        # agentcore
WORKSHOP_TAGS_KV = [{"Key": WORKSHOP_TAG_KEY, "Value": WORKSHOP_TAG_VALUE}]       # ecr
WORKSHOP_TAGS_KV_LOWER = [{"key": WORKSHOP_TAG_KEY, "value": WORKSHOP_TAG_VALUE}] # codebuild

AWS_READY = boto3.Session().get_credentials() is not None
missing = [name for name, value in NEO4J_VALUES.items() if not value]
DEPLOY_READY = bool(GATEWAY_URL and RUNTIME_ROLE_ARN) and AWS_READY and not missing

print(f"region:            {REGION}")
print(f"model:             {MODEL_ID}")
print(f"prefix:            {DEMO06_PREFIX}")
print(f"runtime name:      {RUNTIME_NAME}")
print(f"gateway target:    {GATEWAY_TARGET_NAME}")
print(f"AWS credentials:   {'found' if AWS_READY else 'NOT FOUND'}")
print(f"gateway URL:       {GATEWAY_URL or 'NOT SET'}")
print(f"runtime role:      {RUNTIME_ROLE_ARN or 'NOT SET'}")
print(f"Neo4j values:      {'all four present' if not missing else 'missing ' + ', '.join(missing)}")

if not DEPLOY_READY:
    print()
    print("Not ready to deploy. Every live cell below will skip.")
    if not (GATEWAY_URL and RUNTIME_ROLE_ARN):
        print("  Run the provisioning script first, from the repository root:")
        print("    uv run setup/provision_agentcore.py provision")
        print("  It is idempotent, and it creates real billable resources.")
    if missing:
        print(f"  Add to {REPO_ROOT / '.env'}: {', '.join(missing)}")
    if not AWS_READY:
        print("  Configure AWS credentials for the account that was provisioned.")
else:
    print()
    print("Ready to deploy.")

---
## Step 2: Give the image the shared package

**Open [`deployment-tools/booking_agent.py`](deployment-tools/booking_agent.py)
now and read it before running this step.** It is about 300 lines and it is the
only source file that ships into the container. Everything the deployed agent
does is in there: the two tools it is given, the system prompt, the
`ReservationRequestGuard` hook that pins the caller's `request_id`, the
`CommandResultRecorder` hook that carries the command's verdict back out, and
the `invoke` entrypoint AgentCore Runtime calls. Reading it is what makes the
rest of this notebook a deployment rather than a magic trick.

Its opening import is the one that has to keep working inside a container:

```python
from workshop.hybrid_retrieval import GROUNDING_INSTRUCTIONS, search_hotel_knowledge
```

That is the point of the whole deployment. The retrieval tool the Runtime serves
is the one Lab 2 built, imported rather than reimplemented, so there is no
second copy to drift.

It also means the image needs that package, and the package lives at the
repository root, outside this build context. A container build cannot reach up
out of its own context, so the package is built into a wheel here, next to the
Dockerfile, on every deploy. Building it rather than committing it means the
image always carries the source you have in front of you, including any edit you
made during Lab 4.

In [ ]:
# Resolving the build context is pure path arithmetic, so it runs either way and
# the guarded cells below can rely on DEPLOY_DIR existing.
DEPLOY_DIR = Path.cwd() / "deployment-tools"
if not DEPLOY_DIR.is_dir():
    DEPLOY_DIR = Path.cwd() if Path.cwd().name == "deployment-tools" else DEPLOY_DIR
if not DEPLOY_DIR.is_dir():
    raise FileNotFoundError(
        "deployment-tools/ not found. Run this notebook from 05-agentcore-deploy/."
    )

VENDOR_DIR = DEPLOY_DIR / "vendor"
PACKAGE_DIR = REPO_ROOT / "workshop"
REQUIREMENTS = DEPLOY_DIR / "agent_requirements.txt"

# Everything below this line changes files on disk, and it is only worth doing
# if there is a deploy to do. Without the guard, `--labs 5 --include-deploy`
# against an unprovisioned account fails on a missing `uv` or a wheel mismatch
# instead of skipping, and the failure has nothing to do with the deploy.
if not DEPLOY_READY:
    print("Skipping wheel build: see Step 1.")
else:
    # Remove any earlier wheel first. A stale wheel from a previous version would
    # still satisfy the pinned requirement line and would ship code you no longer
    # have, which is exactly the kind of silent drift this lab is about.
    for stale in VENDOR_DIR.glob("*.whl"):
        stale.unlink()
        print(f"Removed stale wheel: {stale.name}")

    if shutil.which("uv") is None:
        raise RuntimeError(
            "uv is not on PATH, and it is what builds the wheel. Install it from "
            "https://docs.astral.sh/uv/ , or build the wheel by hand with "
            "`python -m build --wheel --outdir deployment-tools/vendor ../workshop`."
        )

    subprocess.run(
        ["uv", "build", "--wheel", "--out-dir", str(VENDOR_DIR), str(PACKAGE_DIR)],
        check=True,
    )

    wheels = sorted(VENDOR_DIR.glob("*.whl"))
    if len(wheels) != 1:
        raise RuntimeError(f"Expected exactly one wheel in {VENDOR_DIR}, found {wheels}")
    wheel = wheels[0]

    # agent_requirements.txt pins the wheel by filename rather than asking the
    # resolver for a package called `workshop`, which would let it reach PyPI for
    # something unrelated. The pin carries the version, so a version bump in
    # workshop/pyproject.toml has to be reflected there too. Fail here, with the
    # line to change, rather than inside a CodeBuild log ten minutes from now.
    pinned = f"./vendor/{wheel.name}"
    if pinned not in REQUIREMENTS.read_text(encoding="utf-8"):
        raise RuntimeError(
            f"{REQUIREMENTS.name} does not pin the wheel that was just built.\n"
            f"  built:    {pinned}\n"
            f"  Update that line in {REQUIREMENTS}, then re-run this cell."
        )

    print(f"\nBuilt {wheel.name} ({wheel.stat().st_size // 1024} KB)")
    print(f"Pinned in {REQUIREMENTS.name} as {pinned}")

---
## Step 3: Pre-flight cleanup

The starter toolkit writes a `.bedrock_agentcore.yaml` beside the entrypoint,
holding the runtime ID from the last deploy. A stale one makes this run try to
update a Runtime that teardown already deleted, so it goes first.

Note what this cell does **not** delete, and why:

- **The CodeBuild project stays.** `launch()` calls
  `create_or_update_project`, so an existing project is updated rather than
  colliding, and the deploy re-runs idempotently without a blind delete. A blind
  delete would also bypass the tag gate every other teardown path here honors.
- **The path is exact and local.** An earlier version globbed
  `~/.bedrock_agentcore*.yaml`. `HOME` is shared with every other AgentCore
  project on the machine, so running this workshop destroyed unrelated local
  config.

In [ ]:
# Guarded for the same reason as the wheel build, and for one more: deleting
# .bedrock_agentcore.yaml when nothing is being deployed throws away the config
# for a Runtime that is still running, which is the one file that records its
# runtime ID. Nothing here should touch disk unless a deploy follows it.
if not DEPLOY_READY:
    print("Skipping pre-flight cleanup: nothing is being deployed.")
else:
    # Configure and launch run from inside deployment-tools/, because that is the
    # container build context. The starter toolkit uses the current directory as
    # the build root: it honors the Dockerfile it finds there, and copies only
    # that directory into the image. Run it from 05-agentcore-deploy/ instead and
    # the toolkit would generate its own Dockerfile, ignore the one written for
    # this agent, and ship the whole lab folder.
    if Path.cwd() != DEPLOY_DIR:
        os.chdir(DEPLOY_DIR)
    print(f"Build context: {Path.cwd()}")

    # Pre-flight cleanup: remove the starter toolkit config from previous runs.
    # The CodeBuild project is intentionally NOT deleted here. The starter
    # toolkit's launch() calls create_or_update_project, which updates an
    # existing project instead of raising ResourceAlreadyExistsException, so the
    # deploy re-runs idempotently without a blind delete. A blind delete_project
    # also bypassed the WorkshopResource tag gate that every other teardown path
    # here honors.

    # Delete starter toolkit config with old runtime ID.
    # Remove only the config file THIS notebook's toolkit run writes, in THIS
    # directory. A previous version globbed ~/.bedrock_agentcore*.yaml. HOME is
    # shared with every other AgentCore project on the machine, so running this
    # workshop destroyed unrelated local config. Exact path, current directory
    # only.
    local_cfg = os.path.join(os.getcwd(), ".bedrock_agentcore.yaml")
    if os.path.exists(local_cfg):
        os.remove(local_cfg)
        print(f"Deleted stale config: {local_cfg}")

    print("Pre-flight cleanup done")

---
## Step 4: Configure and launch

`launch()` ships the build context to CodeBuild, which builds an ARM64 image and
pushes it to ECR, then creates or updates the Runtime from that image. It takes
three to five minutes and prints nothing useful for most of them.

The environment variables are the whole configuration surface of the deployed
agent. `GATEWAY_URL` is how it finds the reservation command; without it,
`booking_agent.py` raises on the first invocation rather than answering as if it
had a tool it does not have. The four `NEO4J_*` values are the read-only
connection the retrieval tool uses. `MODEL_ID` keeps the deployed agent on the
same model the local one used, so a difference in behavior is a difference in
deployment and not a difference in model.

The cell finishes by writing `AGENT_RUNTIME_ARN` into the repository-root
`.env`, using the same in-place upsert `setup/provision_agentcore.py` uses for
the three keys it manages. `5.3_agentcore_walkthrough.ipynb` runs in its own
kernel, so an ARN that only exists as a printed line and a local variable is an
ARN `5.3` cannot see. Writing it means `5.3` works whether a participant opens
it by hand or the acceptance runner opens it unattended.

In [ ]:
from workshop.env_file import update_env_file

# The key 5.3 reads. Provisioning does not manage it, because provisioning does
# not know the ARN: only a launch produces one.
ENV_RUNTIME_ARN_KEY = "AGENT_RUNTIME_ARN"
ENV_HEADER = "# --- Lab 5.1 AgentCore deploy (written by 5.1_agentcore_deploy.ipynb) ---"

# Bound before the branch, and before the launch, on purpose. Step 5 raises a
# readable error when RUNTIME_ARN is empty, and it can only do that if the name
# exists. A launch that raises partway through used to leave the name unbound
# entirely, so the next cell died on a NameError and the guidance never printed.
RUNTIME_ARN = ""
RUNTIME_ID = None

if not DEPLOY_READY:
    print("Skipping launch: see Step 1.")
else:
    from bedrock_agentcore_starter_toolkit import Runtime

    print(f"Role:            {RUNTIME_ROLE_ARN}")
    print(f"Gateway URL:     {GATEWAY_URL}")
    print(f"Gateway target:  {GATEWAY_TARGET_NAME}")

    agent_runtime = Runtime()

    agent_runtime.configure(
        entrypoint="booking_agent.py",
        execution_role=RUNTIME_ROLE_ARN,
        auto_create_ecr=True,
        requirements_file="agent_requirements.txt",
        region=REGION,
        agent_name=RUNTIME_NAME,
        deployment_type="container",
        non_interactive=True,
    )

    print("\nLaunching agent (3-5 minutes)...")

    result = agent_runtime.launch(
        auto_update_on_conflict=True,
        env_vars={
            "AWS_REGION": REGION,
            "GATEWAY_URL": GATEWAY_URL,
            # The other half of the Gateway coordinates. booking_agent.py builds
            # the MCP tool name from this and refuses to run against a Gateway
            # that publishes anything else, so a participant on their own prefix
            # needs it passed in here exactly as provisioning named the target.
            "GATEWAY_TARGET_NAME": GATEWAY_TARGET_NAME,
            "MODEL_ID": MODEL_ID,
            **NEO4J_VALUES,
        },
    )

    RUNTIME_ARN = result.agent_arn
    if not RUNTIME_ARN:
        raise RuntimeError("launch() returned no agent ARN; read the CodeBuild logs.")

    # The trailing segment of the ARN is the runtime ID, and it is what names the
    # CloudWatch log group. Printing it saves a console hunt when the smoke tests
    # below produce something you want to read the logs for.
    RUNTIME_ID = RUNTIME_ARN.split("/")[-1]

    update_env_file(
        REPO_ROOT / ".env",
        {ENV_RUNTIME_ARN_KEY: RUNTIME_ARN},
        header=ENV_HEADER,
    )

    print(f"\nAgent deployed: {RUNTIME_ARN}")
    print(f"Runtime ID:     {RUNTIME_ID}")
    print(f"Log group:      /aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT")
    print(f"\nWrote {ENV_RUNTIME_ARN_KEY} to {REPO_ROOT / '.env'} for 5.3 to read.")
    print("To use it in a shell of your own:")
    print(f"  export {ENV_RUNTIME_ARN_KEY}={RUNTIME_ARN}")

---
## Step 5: Tag the resources the toolkit created

The starter toolkit creates the ECR repository, the CodeBuild project and the
AgentCore Runtime on your behalf, and does not pass the workshop tag through.
`5.2_teardown.ipynb` deletes only tagged resources, so these three have to be
tagged now. Skip this cell and teardown will refuse to delete them, exit
non-zero, and you will keep paying for them.

In [ ]:
if not DEPLOY_READY:
    print("Skipping tagging: nothing was deployed.")
else:
    # --- Tag the resources the starter toolkit created ---
    # The toolkit creates the ECR repo, the CodeBuild project and the AgentCore
    # Runtime itself and does not forward tags, so they are tagged here,
    # immediately after deploy. Cleanup deletes only tagged resources; skip this
    # and teardown will refuse to remove them and will exit non-zero, leaving
    # billable infrastructure running.
    #
    # Every target below is addressed by EXACT name or by ARN. Nothing is
    # enumerated and nothing is prefix-matched. In particular the toolkit's
    # shared AmazonBedrockAgentCoreSDKCodeBuild-* IAM role is deliberately NOT
    # tagged: it is shared across projects, costs nothing, and tagging it would
    # make it eligible for deletion. Deleting roles by name shape once destroyed
    # five unrelated roles in this account.

    ecr_client = boto3.client("ecr", region_name=REGION)
    codebuild_client = boto3.client("codebuild", region_name=REGION)
    agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

    ECR_REPO = f"bedrock-agentcore-{RUNTIME_NAME.lower()}"
    CB_PROJECT = f"bedrock-agentcore-{RUNTIME_NAME.lower()}-builder"

    # ECR repository: resourceArn=, but capitalised {Key, Value} members
    try:
        repo = ecr_client.describe_repositories(repositoryNames=[ECR_REPO])["repositories"][0]
        ecr_client.tag_resource(resourceArn=repo["repositoryArn"], tags=WORKSHOP_TAGS_KV)
        print(f"Tagged ECR repository: {ECR_REPO}")
    except ecr_client.exceptions.RepositoryNotFoundException:
        print(f"ECR repository not found (nothing to tag): {ECR_REPO}")

    # CodeBuild project: lowercase key/value members, applied via update_project.
    # update_project REPLACES the whole tag set, so merge rather than clobber
    # whatever the starter toolkit put there.
    projects = codebuild_client.batch_get_projects(names=[CB_PROJECT])["projects"]
    if projects:
        merged = [t for t in projects[0].get("tags", []) if t.get("key") != WORKSHOP_TAG_KEY]
        codebuild_client.update_project(name=CB_PROJECT, tags=merged + WORKSHOP_TAGS_KV_LOWER)
        print(f"Tagged CodeBuild project: {CB_PROJECT}")
    else:
        print(f"CodeBuild project not found (nothing to tag): {CB_PROJECT}")

    # AgentCore Runtime: tag by the ARN the launch returned
    if not RUNTIME_ARN:
        raise RuntimeError("RUNTIME_ARN is not set. Re-run the launch cell before tagging.")
    agentcore.tag_resource(resourceArn=RUNTIME_ARN, tags=WORKSHOP_TAGS_MAP)
    print(f"Tagged AgentCore Runtime: {RUNTIME_ARN}")

    # Verify rather than trust: read the tags back.
    runtime_tags = agentcore.list_tags_for_resource(resourceArn=RUNTIME_ARN).get("tags", {})
    if runtime_tags.get(WORKSHOP_TAG_KEY) != WORKSHOP_TAG_VALUE:
        raise RuntimeError(f"Runtime tag did not stick. Read back: {runtime_tags}")
    print("\nAll toolkit-created resources tagged and verified.")

---
## Step 6: Four smoke tests

These are the four behaviors Labs 2, 3 and 4 established, asked again over the
network against the deployed Runtime. They are the same four because that is the
test: deployment is supposed to change where the agent runs and nothing about
what it does.

Each one asserts, the way the equivalent case in `4.1_reservation_write.ipynb`
asserts. Printing alone would let a Runtime that answers from the model's own
memory, or a Lambda whose `workshop` import never resolved, produce a clean
notebook. What gets asserted on matters as much as that it is asserted: a model
is free to word an answer however it likes, so nothing below checks a sentence.
The claims are structural, and where a claim is about the write, the graph is
the witness rather than the response text.

| # | Question | What is asserted |
| --- | --- | --- |
| 1 | The hero question | `search_hotel_knowledge` appears in `tools_used`. An answer produced without it is recital |
| 2 | The availability question | Retrieval ran, the tool returned `answerable: false` for `live_room_availability`, and the reservation command was not called |
| 3 | A 15-guest reservation request | The command ran, its own verdict carries `reason_code` `max_guests_exceeded`, and the graph holds no `ReservationRequest` for this `request_id` |
| 4 | The same request delivered twice | The graph holds exactly one accepted `ReservationRequest`, for `MAX_GUESTS` guests |

Test 2 checks the retrieval tool's structured verdict rather than matching model
prose. The verdict travels through `grounding_result`, so the caller can prove
that evidence was consulted and that it lacked the requested live fact.

Test 3 is worth reading closely, because the payload carries three keys that
answer three different questions. `tools_used` says a call was attempted.
`command_result` is the reservation command's own response, computed inside the
Lambda by a rule that read `max_guests` out of the graph. The graph read says
what exists now. Only the second one can tell a rule rejection apart from a
cancelled call or a Lambda that failed on auth, and all three of those leave an
empty graph.

Test 4 writes to your graph. It creates one `ReservationRequest` node linked to
the Cairo hero hotel, the same node Lab 4 created locally.

In [ ]:
import json
import uuid
from datetime import date, timedelta

from workshop.contracts import MAX_GUESTS, OVER_LIMIT_GUESTS, ReservationReason
from workshop.graph_setup import HERO_NAME

# The Gateway form of the reservation command, exactly as booking_agent.py
# names it: the Gateway target name and the schema tool name joined by three
# underscores. Built from the same GATEWAY_TARGET_NAME that Step 4 passed into
# the container, rather than importing booking_agent.py, which would pull the
# Runtime SDK into this kernel to read one string.
COMMAND_TOOL = f"{GATEWAY_TARGET_NAME}___create_reservation_request"

# One caller-created UUID, reused for every delivery of the same reservation
# request. It is the idempotency key and the correlation identifier across
# Runtime, Gateway, the Lambda, and the CloudWatch log lines for all three.
REQUEST_ID = str(uuid.uuid4())

# Relative to today, never a hardcoded date. A fixed future date rots into the
# past and silently flips a passing check-in into a failing one.
CHECK_IN = (date.today() + timedelta(days=30)).isoformat()
CHECK_OUT = (date.today() + timedelta(days=32)).isoformat()

RESERVATION_QUERY = (
    "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
    "RETURN r.status AS status, r.guests AS guests, h.hotel_id AS hotel_id, "
    "toString(r.created_at) AS created_at"
)


def ask(prompt, runtime_arn, request_id=None, session_id=None):
    """Invoke the deployed Runtime once and print what came back.

    `runtime_arn` is a parameter rather than a closure over the module-level
    RUNTIME_ARN, so it is visible at every call site which Runtime is being
    invoked. A helper that silently picks up whatever ARN happens to be bound is
    the kind of thing that quietly keeps working against a Runtime you thought
    you had torn down.

    `grounding_result` and `command_result` are structured tool verdicts carried
    back out of the container. Print both because the model does not get to
    phrase either one.
    """
    payload = {"prompt": prompt}
    if request_id is not None:
        payload["request_id"] = request_id

    client = boto3.client("bedrock-agentcore", region_name=REGION)
    response = client.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn,
        runtimeSessionId=session_id or str(uuid.uuid4()),
        payload=json.dumps(payload).encode("utf-8"),
        qualifier="DEFAULT",
    )
    result = json.loads(response["response"].read())

    print(f"Q: {prompt}\n")
    print(f"A: {result.get('response')}\n")
    print(f"tools used:       {result.get('tools_used') or 'none'}")
    print(f"grounding result: {result.get('grounding_result') or 'none'}")
    print(f"command result:   {result.get('command_result') or 'none'}")
    return result


def reservation_rows(request_id):
    """Read back what the deployed command actually wrote to the graph.

    The response text is the model's account of what happened. This is the
    graph's. Only the second one settles whether a node exists, so the write
    assertions below use it: no phrasing of an answer can make a row appear or
    disappear.
    """
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(
        NEO4J_VALUES["NEO4J_URI"],
        auth=(NEO4J_VALUES["NEO4J_USERNAME"], NEO4J_VALUES["NEO4J_PASSWORD"]),
    )
    try:
        with driver.session(database=NEO4J_VALUES["NEO4J_DATABASE"]) as session:
            return [dict(record) for record in session.run(RESERVATION_QUERY, rid=request_id)]
    finally:
        driver.close()


if DEPLOY_READY:
    print(f"request_id: {REQUEST_ID}")
    print(f"stay:       {CHECK_IN} to {CHECK_OUT}")
else:
    print("Skipping smoke tests: nothing was deployed.")

In [ ]:
# Test 1: the hero question. Grounded retrieval over the network.
if not DEPLOY_READY:
    print("Skipped.")
else:
    result = ask(f"What amenities and guest rating does {HERO_NAME} have?", RUNTIME_ARN)

    # The deployed agent has to reach the graph to answer this. An answer with
    # an empty tools_used came out of the model's own memory, which is the exact
    # failure the workshop exists to stop, and it is also what a broken
    # `workshop` import inside the image looks like from out here.
    assert "search_hotel_knowledge" in result["tools_used"], (
        "the deployed agent answered without calling the retrieval tool: "
        f"tools_used={result['tools_used']!r}"
    )
    assert result["response"].strip(), "the Runtime returned an empty response"
    print("\nPASS: answered through the Lab 2 retriever running inside the Runtime.")

In [ ]:
# Test 2: the availability question. The graph holds no live availability, so
# the correct answer is to decline rather than to invent one.
if not DEPLOY_READY:
    print("Skipped.")
else:
    result = ask(f"Does {HERO_NAME} guarantee room availability next weekend?", RUNTIME_ARN)

    # Assert the evidence path and its structured verdict. No assertion depends
    # on the model choosing a particular refusal sentence.
    assert "search_hotel_knowledge" in result["tools_used"], (
        "the deployed agent answered without calling the retrieval tool: "
        f"tools_used={result['tools_used']!r}"
    )
    grounding = result.get("grounding_result")
    assert grounding is not None, "retrieval ran but returned no grounding verdict"
    assert grounding.get("answerable") is False, grounding
    assert grounding.get("missing_fact") == "live_room_availability", grounding
    assert grounding.get("evidence_ids"), grounding
    assert COMMAND_TOOL not in result["tools_used"], (
        "an availability question reached the reservation command: "
        f"tools_used={result['tools_used']!r}"
    )
    assert result["response"].strip(), "the Runtime returned an empty response"
    print(
        "\nPASS: retrieval ran, reported answerable false for live room "
        "availability, and no write was attempted."
    )

In [ ]:
# Test 3: an over-limit reservation request. The max_guests rule lives in the
# graph and the command reads it, so the rejection happens inside the same
# boundary as the write. Nothing is written.
if not DEPLOY_READY:
    print("Skipped.")
else:
    result = ask(
        f"Find {HERO_NAME} and create a reservation request for "
        f"{OVER_LIMIT_GUESTS} guests, check-in {CHECK_IN}, check-out {CHECK_OUT}.",
        RUNTIME_ARN,
        request_id=REQUEST_ID,
    )

    # Three facts, and all three have to hold.
    #
    # The middle one is the one that makes this test mean what the heading says.
    # `tools_used` records that a call was attempted, so a call the container's
    # own guard hook cancelled, a Lambda that failed on auth, and a Gateway 5xx
    # all put the tool name in that list and all leave the graph empty. Those
    # three and a genuine rule rejection are indistinguishable from the first
    # and third assertions alone. `command_result` is the command's own response,
    # computed by the rule reading max_guests out of the graph, so asserting on
    # its reason_code is what separates "the graph refused it" from "something
    # broke between the agent and the graph."
    assert COMMAND_TOOL in result["tools_used"], (
        "the reservation command was never called, so nothing was actually "
        f"tested: tools_used={result['tools_used']!r}"
    )
    verdict = result.get("command_result")
    assert verdict is not None, (
        "the command was attempted but returned no verdict, which means the "
        "call was cancelled or failed before the rule ever ran"
    )
    assert verdict.get("reason_code") == ReservationReason.MAX_GUESTS_EXCEEDED.value, (
        "the request was refused, but not by the max_guests rule in the graph: "
        f"command_result={verdict!r}"
    )
    rows = reservation_rows(REQUEST_ID)
    assert rows == [], (
        f"an over-limit request must write nothing, found {len(rows)} row(s): {rows}"
    )
    print(
        f"\nPASS: the command ran, the graph rule refused {OVER_LIMIT_GUESTS} "
        f"guests with {verdict['reason_code']}, and nothing was written."
    )

In [ ]:
# Test 4: a corrected request within the limit, delivered twice with the same
# request_id. The second delivery returns the existing record rather than
# creating a second one. Idempotence is the command's job, not the model's.
if not DEPLOY_READY:
    print("Skipped.")
else:
    prompt = (
        f"Find {HERO_NAME} and create a reservation request for "
        f"{MAX_GUESTS} guests, check-in {CHECK_IN}, check-out {CHECK_OUT}."
    )
    print("=== first delivery ===")
    first = ask(prompt, RUNTIME_ARN, request_id=REQUEST_ID)
    print("\n=== second delivery, same request_id ===")
    ask(prompt, RUNTIME_ARN, request_id=REQUEST_ID)

    assert COMMAND_TOOL in first["tools_used"], (
        f"the reservation command was never called: tools_used={first['tools_used']!r}"
    )

    # Two deliveries, one node. Whether the model reports the second as a
    # duplicate is up to the model; whether a second node exists is not. Count
    # the rows.
    rows = reservation_rows(REQUEST_ID)
    assert len(rows) == 1, f"expected exactly one request, found {len(rows)}: {rows}"
    assert rows[0]["status"] == "accepted", rows[0]
    assert rows[0]["guests"] == MAX_GUESTS, rows[0]
    print(f"\n{rows[0]}")
    print(
        "\nPASS: two deliveries of the same request_id left exactly one accepted "
        "record in the graph."
    )

---
## What is running now

An AgentCore Runtime, an ECR repository holding its image, and a CodeBuild
project that built it. All three are tagged, which is the only reason
`5.2_teardown.ipynb` can delete them.

## Where the request went

The `request_id` you generated above appears in the Runtime logs, in the
Gateway's record of the tool call, in the reservation Lambda's logs, and on the
`ReservationRequest` node in your graph. One identifier, created by the caller,
carried end to end. That is what makes a deployed agent's behavior something you
can audit after the fact rather than something you have to reproduce.

Runtime logs are in CloudWatch under `/aws/bedrock-agentcore/runtimes/`, in the
log group Step 4 printed.

## Next

- **`5.3_agentcore_walkthrough.ipynb`** (optional) works through the rejection,
  the correction, the graph inspection, and the log correlation one at a time.
  It reads `AGENT_RUNTIME_ARN`, which Step 4 wrote into the repository-root
  `.env`, so it works in a fresh kernel with nothing further to do. Step 4 also
  printed the `export` line if you would rather set it in a shell.
- **`5.2_teardown.ipynb`** deletes what this notebook created. Run it before you
  stop. It is the only thing in the workshop that does. After teardown, the
  `AGENT_RUNTIME_ARN` in your `.env` points at a Runtime that no longer exists;
  re-running this notebook overwrites it.